# Test Functionality of GestionOt{class}

### Pasos para calificar una actividad.

1. Separar los eventos que tienen alimentador de los que no.
   
   1.1. Separar y calificar aquellos que son de TRANSPORTE, ALIMENTACIÓN, SE LABORA, INFO, se repite en la calificación
   
2. A los eventos que si tienen alimentador.
   
   2.1. Separar aquellos que sabemos que son SAPG, los más fáciles de identificar.

   2.2. Separar aquellos que son de Servicios Ocasionales.

   2.3. Calificar usando la Red Neuronal.

In [1]:

from eerssa import gestionOT
from eerssa import matrizActividades
from pprint import pprint
from pathlib import Path
import pandas as pd

test_path = '/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/'
list_pdfs = []
for path in Path( test_path ).glob("**/*.pmatriz_test"):
  list_pdfs.append( str(path) )
  list_pdfs.sort()

obj_lists = []
for file in list_pdfs:
  ot = gestionOT.GestionOt( file )
  ot.load_ot()
  obj_lists.append( ot )


#matriz_test = matrizActividades.ConvertirOT_a_ActividadesCSV(  obj_lists[3] )


Success!!!


In [48]:
nro_ot = 1
obj_lists[nro_ot].data['link']

'/home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/tests/ot_test/11_LM_Tres_hojas.pdf'

In [49]:
obj_lists[nro_ot].log

[{'t': '2025-01-17T09:32:55.444232',
  'level': 'INFO',
  'message': 'Creacion de la OT',
  'detail': 'Ninguno'}]

In [65]:
matriz_test = matrizActividades.ConvertirOT_a_ActividadesCSV(  obj_lists[nro_ot] )
matriz_test[['Cuenta','Evento','Tipo','Actividad','Alimentador','Fecha']]

,Cuenta,Evento,Tipo,Actividad,Alimentador,Fecha
0,·,En la agencia de la EERSSA El Pangui se coordi...,RUTINARIA,PROG,·,2024-07-04 00:00:00
1,·,"Pangui en el centro de la ciudad, calle Quito ...",CORRECTIVO,NO PROG,El Pangui,2024-07-04 00:00:00
3,·,RECLAMO No. 1100629005 03-07-24/12:05. Pangui ...,CORRECTIVO,PROG,El Pangui,2024-07-04 00:00:00
6,·,RECLAMO No. 1100629005 03-07-24/12:05. Pangui ...,CORRECTIVO,PROG,El Pangui,2024-07-04 00:00:00
9,·,Nos trasladamos desde el Pangui hacia Pachicutza.,TRANSPORTE,TRANSP,·,2024-07-04 00:00:00
10,·,En el sector de Pachicutza junto al parterre d...,PREVENTIVO,PROG,Los Encuentros,2024-07-04 00:00:00
11,·,Nos trasladamos desde el sector de Pachicutza ...,TRANSPORTE,TRANSP,·,2024-07-04 00:00:00
12,·,"En el sector El Padmi, en la est. No. 93779, s...",CORRECTIVO,PROG,Los Encuentros,2024-07-04 00:00:00
13,·,En el sector de Los Encuentros est. No. 185417...,CORRECTIVO,PROG,Los Encuentros,2024-07-04 00:00:00
14,·,Lunch el sector de Los Encuentros.,ALIMENTACI,ALIMEN,·,2024-07-04 00:00:00


In [68]:
matriz_test.loc[ matriz_test['Tipo'] == "TRANSPORTE", 'Cuenta' ] = "transporte"
matriz_test.loc[ matriz_test['Tipo'] == "ALIMENTACI", 'Cuenta' ] = "lunch"
matriz_test.loc[ matriz_test['Actividad'] == "LABORA", 'Cuenta' ] = "se_labora"

In [79]:
matriz_test[['Cuenta','Evento','Tipo','Alimentador','Fecha']]

,Cuenta,Evento,Tipo,Alimentador,Fecha
0,·,En la agencia de la EERSSA El Pangui se coordi...,RUTINARIA,·,2024-07-04 00:00:00
1,·,"Pangui en el centro de la ciudad, calle Quito ...",CORRECTIVO,El Pangui,2024-07-04 00:00:00
3,·,RECLAMO No. 1100629005 03-07-24/12:05. Pangui ...,CORRECTIVO,El Pangui,2024-07-04 00:00:00
6,·,RECLAMO No. 1100629005 03-07-24/12:05. Pangui ...,CORRECTIVO,El Pangui,2024-07-04 00:00:00
9,transporte,Nos trasladamos desde el Pangui hacia Pachicutza.,TRANSPORTE,·,2024-07-04 00:00:00
10,·,En el sector de Pachicutza junto al parterre d...,PREVENTIVO,Los Encuentros,2024-07-04 00:00:00
11,transporte,Nos trasladamos desde el sector de Pachicutza ...,TRANSPORTE,·,2024-07-04 00:00:00
12,·,"En el sector El Padmi, en la estructura. No. 9...",CORRECTIVO,Los Encuentros,2024-07-04 00:00:00
13,·,En el sector de Los Encuentros estructura. No....,CORRECTIVO,Los Encuentros,2024-07-04 00:00:00
14,lunch,Lunch el sector de Los Encuentros.,ALIMENTACI,·,2024-07-04 00:00:00


In [70]:
import re

def limpiar_texto_actividad( actividad ):
  palabras = {
          "#"             : "nro",
          "Est"           : "estructura",
          "Est."          : "estructura",
          "Estr"          : "estructura",
          "est"           : "estructura",
          "est."          : "estructura",
          "est."          : "estructura",
          "estructuras"   : "estructura",
          "poste"         : "estructura",
          "tiraf"         : "tirafusible",
          "med."          : "medidor",
          "med"           : "medidor",
          "med"           : "medidor",
          "CC"            : "Centro de Control",
          "C.C"           : "Centro de Control",
          "C.C."          : "Centro de Control",
          "C C"           : "Centro de Control",
          "trafo"         : "transformador",
          "tranfo"        : "transformador",
          "breiker"       : "breaker",
          "  "            : " ",
  }

  keys = (re.escape(k) for k in palabras.keys())
  pattern = re.compile(r'\b(' + '|'.join(keys) + r')\b')

  resultado = pattern.sub(lambda x: palabras[x.group()], actividad )

  return resultado


In [78]:
matriz_test.loc[:,'Evento'] = matriz_test['Evento'].apply(lambda x:limpiar_texto_actividad(x))

In [77]:
limpiar_texto_actividad("""En el sector El Padmi, en la est. No. 93779, se revisa 1 luminaria de
100 W Na. Apagada. Se ocupa: fotocélula. trafo, tranfo, poste, CC""")

'En el sector El Padmi, en la estructura. No. 93779, se revisa 1 luminaria de\n100 W Na. Apagada. Se ocupa: fotocélula. transformador, transformador, estructura, Centro de Control'

In [18]:
test = obj_lists[nro_ot].data['actividades']
test

[{'Item': '1',
  'Actividad': 'PROG',
  'Evento': 'Bodega Coordinación de trabajos. Se carga materiales y herramientas\nen el vehículo',
  'Ali': None,
  'Alimentador': None,
  'Tipo': 'PREDICTIVO',
  'InicioEvento': '2022-07-18 08:00:00',
  'FinEvento': '2022-07-18 08:30:00'},
 {'Item': '2',
  'Actividad': 'TRANSP',
  'Evento': 'Daño Centro de Control (mensaje WhatsApp)\nNos trasladamos de Guayzimi a Shakay',
  'Ali': None,
  'Alimentador': None,
  'Tipo': 'TRANSPORTE',
  'InicioEvento': '2022-07-18 08:30:00',
  'FinEvento': '2022-07-18 10:40:00'},
 {'Item': '3',
  'Actividad': 'NO PROG',
  'Evento': 'Sector Shakay, con apoyo de canoa a motor GAD-Provincial cruzamos el\nRío Nangaritza a barrio Shakay, poste 501256 transformador 5KVA sicap\n15004 abierto',
  'Ali': 'ALI',
  'Alimentador': 'Paquisha',
  'Tipo': 'CORRECTIVO',
  'InicioEvento': '2022-07-18 10:40:00',
  'FinEvento': '2022-07-18 14:00:00'},
 {'Item': '4',
  'Actividad': None,
  'Evento': 'Por falla poste S/N (500860) aislad

## Probar DASK

## Test dierctory

In [3]:
from dask.distributed import LocalCluster
client = LocalCluster().get_client()


In [7]:
## ¿Como se usa DASK para creacion de objetos?

ots_raw = []
for file in list_pdfs:
  this_ot = client.submit( gestionOT.GestionOt, file )
  conversion = client.submit( gestionOT.GestionOt, file )

In [3]:
obj_lists = []
obj_data  = []
for file in list_pdfs:
  ot = gestionOT.GestionOt( file )
  ot.load_ot()
  obj_lists.append( ot )
  obj_data.append( ot.data )

In [5]:
len(obj_data)

29

In [4]:
obj_lists[3].log

[{'t': '2025-01-17T03:49:16.915439',
  'level': 'INFO',
  'message': 'Creacion de la OT',
  'detail': 'Ninguno'}]

In [4]:
ot_as_pd = pd.DataFrame( obj_data )

In [15]:
dbg

version                                                     0.12.0
link             /home/vlad/GIT/eerssa_gh/ordenes_de_trabajo/te...
id_ot                                                     134508.0
exito                                                         True
cuadrilla                          Yacuambi Z1 (Cuadrilla. Nro. 8)
responsable                     [LOZANO SIGCHO NAUN ENRIQUE, JECE]
colaboradores    {'total': 3, 'nombres': [['CABRERA GONZALEZ LU...
diaSemana                                                    lunes
fecha                                    2024-07-29 00:00:00-05:00
fechaInicio                            lunes, 29 de julio del 2024
fechaFinal                                     29/07/2024 20:40:00
sitio                 Yacuambi - Tamboloma, Hucapamba y Jembuentza
descripcion      Traslado a Tamboloma para revisar sector sin s...
tEstimado                                                        8
vehiculo         {'numero': 'R-171', 'placa': 'AAA-4278', 'mar